
# BDA501 — IEEE-CIS Fraud Detection: Spark EDA, Imbalance Handling, and Training Data Pipeline

This notebook is a complete, reproducible **Apache Spark** pipeline for the BDA501 group final assignment. It:

1. reads the IEEE-CIS transaction and identity CSV files from the exact Windows project structure or from Docker-mounted paths;
2. validates dataset size, schema, keys, and transaction–identity joins;
3. performs distributed profiling and risk-focused EDA;
4. cleans categorical values and engineers transaction, card, email, device, time, and missingness features;
5. uses a chronological 70/15/15 split based on `TransactionDT`;
6. fits all imputations and aggregate lookups on the training period only;
7. creates original, class-weighted, and controlled-undersampled training datasets;
8. leaves validation and holdout distributions untouched;
9. exports training-ready Parquet datasets and reports;
10. trains a simple Spark MLlib Decision Tree for the course demonstration;
11. saves representative true-positive, true-negative, false-positive, and false-negative demo cases.

## Path contract

Local Windows input:

```text
D:\MSE\16. Big Data\Fraud-Detection-Score-Risk\data\data\ieee-fraud-detection
```

Docker input mount:

```text
/app/data/raw
```

Default processed output:

```text
<project-root>/data/processed/ieee_cis_spark
```

The notebook creates missing output folders automatically. Raw Kaggle files are never modified.


## One-time local environment bootstrap (Windows / VS Code / Jupyter)

This cell installs PySpark into the **same Python interpreter used by the active notebook kernel**. It also validates Java 17, which Spark requires. In Docker, the dependencies are already installed, so the cell only performs validation.


In [ ]:
# Run this cell before the imports cell.
# It fixes the common case where PySpark was installed in .venv but VS Code/Jupyter selected another kernel.

from __future__ import annotations

import glob
import importlib
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

PYSPARK_VERSION = os.getenv("PYSPARK_VERSION", "3.5.1")
AUTO_INSTALL_PYSPARK = os.getenv("AUTO_INSTALL_PYSPARK", "1") == "1"

print("Active notebook Python:", sys.executable)
print("Python version:", sys.version.split()[0])

# Keep Spark workers on the exact interpreter used by this notebook.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")


def install_pyspark_into_active_kernel() -> None:
    if importlib.util.find_spec("pyspark") is not None:
        return
    if not AUTO_INSTALL_PYSPARK:
        raise ModuleNotFoundError(
            "PySpark is missing and AUTO_INSTALL_PYSPARK=0. Install it with: "
            f'"{sys.executable}" -m pip install pyspark=={PYSPARK_VERSION}'
        )

    print(f"PySpark is missing. Installing pyspark=={PYSPARK_VERSION} into the active kernel...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        f"pyspark=={PYSPARK_VERSION}",
    ])
    importlib.invalidate_caches()


def discover_java_home() -> str | None:
    configured = os.environ.get("JAVA_HOME")
    if configured and (Path(configured) / "bin" / ("java.exe" if os.name == "nt" else "java")).exists():
        return configured

    java_cmd = shutil.which("java")
    if java_cmd:
        # JAVA_HOME is the parent of bin/java(.exe).
        return str(Path(java_cmd).resolve().parent.parent)

    if os.name == "nt":
        patterns = [
            r"C:\Program Files\Eclipse Adoptium\jdk-17*",
            r"C:\Program Files\Java\jdk-17*",
            r"C:\Program Files\Microsoft\jdk-17*",
            r"C:\Program Files\Amazon Corretto\jdk17*",
        ]
        candidates = []
        for pattern in patterns:
            candidates.extend(glob.glob(pattern))
        candidates = [Path(p) for p in candidates if (Path(p) / "bin" / "java.exe").exists()]
        if candidates:
            return str(sorted(candidates)[-1])
    return None


install_pyspark_into_active_kernel()

JAVA_HOME = discover_java_home()
if not JAVA_HOME:
    raise RuntimeError(
        "Java was not found. Apache Spark requires Java 17 for this project.\n\n"
        "Windows PowerShell installation command:\n"
        "  winget install EclipseAdoptium.Temurin.17.JDK\n\n"
        "After installation, completely restart VS Code/Jupyter and run this notebook again. "
        "Docker users do not need to install Java on the host because Java is included in the image."
    )

os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["PATH"] = str(Path(JAVA_HOME) / "bin") + os.pathsep + os.environ.get("PATH", "")

java_check = subprocess.run(
    [str(Path(JAVA_HOME) / "bin" / ("java.exe" if os.name == "nt" else "java")), "-version"],
    capture_output=True,
    text=True,
)
if java_check.returncode != 0:
    raise RuntimeError(f"Java validation failed for JAVA_HOME={JAVA_HOME}: {java_check.stderr}")

import pyspark

print("PySpark version:", pyspark.__version__)
print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("Environment bootstrap completed successfully.")


## Pipeline imports and configuration


In [ ]:
from __future__ import annotations

import csv
import json
import logging
import os
import platform
import re
import sys
import time
from functools import reduce
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = None
    display = print

try:
    from pyspark import StorageLevel
    from pyspark.ml import Pipeline
    from pyspark.ml.classification import DecisionTreeClassifier
    from pyspark.ml.evaluation import BinaryClassificationEvaluator
    from pyspark.ml.feature import Imputer, StringIndexer, VectorAssembler
    from pyspark.ml.functions import vector_to_array
    from pyspark.ml.pipeline import PipelineModel
    from pyspark.sql import DataFrame, SparkSession, Window
    from pyspark.sql import functions as F
    from pyspark.sql import types as T
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "PySpark is not installed in the active environment. Install the requirements or run the supplied Docker image."
    ) from exc

SEED = int(os.getenv("PIPELINE_SEED", "42"))
np.random.seed(SEED)

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("ieee_cis_bda501")

WINDOWS_PROJECT_ROOT = Path(r"D:\MSE\16. Big Data\Fraud-Detection-Score-Risk")
WINDOWS_RAW_DATA_DIR = Path(r"D:\MSE\16. Big Data\Fraud-Detection-Score-Risk\data\data\ieee-fraud-detection")
REQUIRED_FILES = [
    "train_transaction.csv",
    "train_identity.csv",
    "test_transaction.csv",
    "test_identity.csv",
]
OPTIONAL_FILES = ["sample_submission.csv"]


def _existing_path(value: str | None) -> Path | None:
    if not value:
        return None
    path = Path(value).expanduser()
    return path.resolve() if path.exists() else None


def resolve_project_root() -> Path:
    env_root = _existing_path(os.getenv("PROJECT_ROOT"))
    if env_root:
        return env_root
    if os.name == "nt" and WINDOWS_PROJECT_ROOT.exists():
        return WINDOWS_PROJECT_ROOT.resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / ".git").exists() or (candidate / "docker-compose.yml").exists() or (candidate / "docker-compose.preprocessing.yml").exists():
            return candidate
    return cwd


def contains_required_files(directory: Path) -> bool:
    return directory.is_dir() and all((directory / name).is_file() for name in REQUIRED_FILES)


def resolve_raw_data_dir(project_root: Path) -> Path:
    candidates: list[Path] = []
    env_raw = os.getenv("IEEE_CIS_DATA_DIR")
    if env_raw:
        candidates.append(Path(env_raw).expanduser())
    if os.name == "nt":
        candidates.append(WINDOWS_RAW_DATA_DIR)
    candidates.extend([
        project_root / "data" / "data" / "ieee-fraud-detection",
        project_root / "data" / "ieee-fraud-detection",
        project_root / "data" / "raw" / "ieee-fraud-detection",
        project_root / "data" / "raw",
        Path("/app/data/raw"),
    ])
    for candidate in candidates:
        candidate = candidate.resolve()
        if contains_required_files(candidate):
            return candidate
    preferred = candidates[0].resolve() if candidates else (project_root / "data" / "data" / "ieee-fraud-detection").resolve()
    return preferred


def resolve_output_dir(project_root: Path) -> Path:
    env_output = os.getenv("IEEE_CIS_OUTPUT_DIR")
    output = Path(env_output).expanduser() if env_output else project_root / "data" / "processed" / "ieee_cis_spark"
    output = output.resolve()
    output.mkdir(parents=True, exist_ok=True)
    return output


PROJECT_ROOT = resolve_project_root()
RAW_DATA_DIR = resolve_raw_data_dir(PROJECT_ROOT)
OUTPUT_DIR = resolve_output_dir(PROJECT_ROOT)
REPORTS_DIR = OUTPUT_DIR / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
FEATURE_STORE_DIR = OUTPUT_DIR / "feature_store"
MODEL_READY_DIR = OUTPUT_DIR / "model_ready"
MODEL_DIR = OUTPUT_DIR / "artifacts" / "decision_tree_demo"
DEMO_DIR = OUTPUT_DIR / "demo"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
SPARK_LOCAL_DIR = OUTPUT_DIR / "spark-local"
for directory in [REPORTS_DIR, FIGURES_DIR, FEATURE_STORE_DIR, MODEL_READY_DIR, MODEL_DIR.parent, DEMO_DIR, CHECKPOINT_DIR, SPARK_LOCAL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RUN_FULL_PROFILE = os.getenv("RUN_FULL_PROFILE", "true").lower() in {"1", "true", "yes"}
RUN_MODEL_DEMO = os.getenv("RUN_MODEL_DEMO", "true").lower() in {"1", "true", "yes"}
WRITE_WIDE_FEATURE_STORE = os.getenv("WRITE_WIDE_FEATURE_STORE", "true").lower() in {"1", "true", "yes"}
IMBALANCE_RATIO = float(os.getenv("IMBALANCE_RATIO", "4.0"))
PROFILE_BATCH_SIZE = int(os.getenv("PROFILE_BATCH_SIZE", "40"))

print(json.dumps({
    "project_root": str(PROJECT_ROOT),
    "raw_data_dir": str(RAW_DATA_DIR),
    "output_dir": str(OUTPUT_DIR),
    "run_full_profile": RUN_FULL_PROFILE,
    "run_model_demo": RUN_MODEL_DEMO,
    "write_wide_feature_store": WRITE_WIDE_FEATURE_STORE,
    "imbalance_ratio_legit_to_fraud": IMBALANCE_RATIO,
    "seed": SEED,
}, indent=2))

## Spark session and utility functions

In [ ]:
def create_spark_session() -> SparkSession:
    master = os.getenv("SPARK_MASTER", "local[*]")
    shuffle_partitions = os.getenv("SPARK_SHUFFLE_PARTITIONS", str(max(16, (os.cpu_count() or 4) * 2)))
    builder = (
        SparkSession.builder
        .appName("BDA501-IEEE-CIS-Fraud-Preprocessing")
        .master(master)
        .config("spark.sql.shuffle.partitions", shuffle_partitions)
        .config("spark.default.parallelism", os.getenv("SPARK_DEFAULT_PARALLELISM", str(max(8, os.cpu_count() or 4))))
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")
        .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
        .config("spark.local.dir", str(SPARK_LOCAL_DIR))
    )
    spark = builder.getOrCreate()
    spark.sparkContext.setLogLevel(os.getenv("SPARK_LOG_LEVEL", "WARN"))
    spark.sparkContext.setCheckpointDir(str(CHECKPOINT_DIR))
    return spark


spark = create_spark_session()
print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("Spark UI:", spark.sparkContext.uiWebUrl or "not available")


def chunked(values: list[str], size: int) -> Iterable[list[str]]:
    for index in range(0, len(values), size):
        yield values[index:index + size]


def spark_path(path: Path) -> str:
    path = path.resolve()
    if os.name == "nt":
        return path.as_uri()
    return str(path)


def read_csv_header(path: Path) -> list[str]:
    with path.open("r", encoding="utf-8", newline="") as handle:
        return next(csv.reader(handle))


def validate_source_files(raw_dir: Path) -> dict[str, Path]:
    missing = [name for name in REQUIRED_FILES if not (raw_dir / name).is_file()]
    if missing:
        expected = "\n".join(f"  - {raw_dir / name}" for name in REQUIRED_FILES)
        raise FileNotFoundError(
            f"Missing IEEE-CIS files: {missing}\nExpected files:\n{expected}\n"
            "For Docker, run from the project root so ./data/data/ieee-fraud-detection is mounted to /app/data/raw."
        )
    result = {Path(name).stem: raw_dir / name for name in REQUIRED_FILES}
    for name in OPTIONAL_FILES:
        if (raw_dir / name).is_file():
            result[Path(name).stem] = raw_dir / name
    return result


def write_json(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")


def write_single_csv(df: DataFrame, path: Path) -> None:
    df.coalesce(1).write.mode("overwrite").option("header", True).csv(spark_path(path))


def write_parquet(df: DataFrame, path: Path, partition_cols: list[str] | None = None) -> None:
    writer = df.write.mode("overwrite")
    if partition_cols:
        writer.partitionBy(*partition_cols).parquet(spark_path(path))
    else:
        writer.parquet(spark_path(path))

## Schemas, ingestion, inventory, and merge audit

In [ ]:
def build_transaction_schema(columns: list[str]) -> T.StructType:
    categorical = {
        "ProductCD", "card1", "card2", "card3", "card4", "card5", "card6", "addr1", "addr2",
        "P_emaildomain", "R_emaildomain", "M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9",
    }
    fields: list[T.StructField] = []
    for name in columns:
        if name == "TransactionID":
            dtype = T.LongType()
        elif name == "isFraud":
            dtype = T.IntegerType()
        elif name == "TransactionDT":
            dtype = T.LongType()
        elif name == "TransactionAmt" or name in {"dist1", "dist2"} or re.fullmatch(r"[CDV]\d+", name):
            dtype = T.DoubleType()
        elif name in categorical:
            dtype = T.StringType()
        else:
            dtype = T.StringType()
        fields.append(T.StructField(name, dtype, True))
    return T.StructType(fields)


def build_identity_schema(columns: list[str]) -> T.StructType:
    categorical = {
        "id_12", "id_15", "id_16", "id_23", "id_27", "id_28", "id_29", "id_30", "id_31",
        "id_33", "id_34", "id_35", "id_36", "id_37", "id_38", "DeviceType", "DeviceInfo",
    }
    fields: list[T.StructField] = []
    for name in columns:
        if name == "TransactionID":
            dtype = T.LongType()
        elif name in categorical:
            dtype = T.StringType()
        elif name.startswith("id_"):
            dtype = T.DoubleType()
        else:
            dtype = T.StringType()
        fields.append(T.StructField(name, dtype, True))
    return T.StructType(fields)


def read_with_schema(path: Path, schema: T.StructType) -> DataFrame:
    return (
        spark.read.format("csv")
        .option("header", True)
        .option("mode", "PERMISSIVE")
        .option("nullValue", "")
        .option("nanValue", "NaN")
        .option("ignoreLeadingWhiteSpace", True)
        .option("ignoreTrailingWhiteSpace", True)
        .schema(schema)
        .load(spark_path(path))
    )


def audit_key(df: DataFrame, dataset_name: str) -> dict[str, object]:
    row = df.agg(
        F.count("*").alias("rows"),
        F.count("TransactionID").alias("non_null_keys"),
        F.countDistinct("TransactionID").alias("distinct_keys"),
    ).first()
    rows = int(row["rows"])
    non_null = int(row["non_null_keys"])
    distinct_keys = int(row["distinct_keys"])
    return {
        "dataset": dataset_name,
        "rows": rows,
        "null_transaction_ids": rows - non_null,
        "duplicate_transaction_ids": rows - distinct_keys,
        "status": "pass" if rows == non_null == distinct_keys else "fail",
    }


def left_join_with_identity(transaction_df: DataFrame, identity_df: DataFrame, dataset_name: str) -> tuple[DataFrame, dict[str, object]]:
    marker = identity_df.select("TransactionID").distinct().withColumn("__has_identity", F.lit(1))
    joined = (
        transaction_df
        .join(identity_df, on="TransactionID", how="left")
        .join(marker, on="TransactionID", how="left")
        .withColumn("has_identity", F.coalesce(F.col("__has_identity"), F.lit(0)).cast("int"))
        .drop("__has_identity")
    )
    before = transaction_df.count()
    after = joined.count()
    matched = joined.filter(F.col("has_identity") == 1).count()
    audit = {
        "dataset": dataset_name,
        "transaction_rows_before": before,
        "identity_rows": identity_df.count(),
        "joined_rows_after": after,
        "matched_identity_rows": matched,
        "unmatched_transaction_rows": after - matched,
        "row_difference": after - before,
        "status": "pass" if after == before else "fail",
    }
    return joined, audit


source_paths = validate_source_files(RAW_DATA_DIR)
inventory_pdf = pd.DataFrame([
    {
        "filename": path.name,
        "path": str(path),
        "size_mb": round(path.stat().st_size / 1024**2, 2),
    }
    for path in source_paths.values()
])
print(inventory_pdf.to_string(index=False))
print(f"Total source size: {inventory_pdf['size_mb'].sum():,.2f} MB")
assert inventory_pdf.loc[inventory_pdf["filename"].isin(REQUIRED_FILES), "size_mb"].sum() >= 500, "The four required files must total at least 500 MB for the BDA501 requirement."

train_tx_schema = build_transaction_schema(read_csv_header(source_paths["train_transaction"]))
test_tx_schema = build_transaction_schema(read_csv_header(source_paths["test_transaction"]))
train_id_schema = build_identity_schema(read_csv_header(source_paths["train_identity"]))
test_id_schema = build_identity_schema(read_csv_header(source_paths["test_identity"]))

train_transaction = read_with_schema(source_paths["train_transaction"], train_tx_schema)
test_transaction = read_with_schema(source_paths["test_transaction"], test_tx_schema)
train_identity = read_with_schema(source_paths["train_identity"], train_id_schema)
test_identity = read_with_schema(source_paths["test_identity"], test_id_schema)

key_audit_rows = [
    audit_key(train_transaction, "train_transaction"),
    audit_key(test_transaction, "test_transaction"),
    audit_key(train_identity, "train_identity"),
    audit_key(test_identity, "test_identity"),
]
key_audit = spark.createDataFrame(pd.DataFrame(key_audit_rows))
key_audit.show(truncate=False)
assert key_audit.filter(F.col("status") == "fail").count() == 0, "TransactionID validation failed."

train_merged, train_join_audit = left_join_with_identity(train_transaction, train_identity, "train")
test_merged, test_join_audit = left_join_with_identity(test_transaction, test_identity, "test")
join_audit = spark.createDataFrame(pd.DataFrame([train_join_audit, test_join_audit]))
join_audit.show(truncate=False)
assert join_audit.filter(F.col("status") == "fail").count() == 0, "Transaction-identity join changed the transaction grain."

train_merged = train_merged.persist(StorageLevel.MEMORY_AND_DISK)
test_merged = test_merged.persist(StorageLevel.MEMORY_AND_DISK)
train_rows = train_merged.count()
test_rows = test_merged.count()
print("Merged train rows/columns:", train_rows, len(train_merged.columns))
print("Merged test rows/columns:", test_rows, len(test_merged.columns))

## Distributed data profiling and data-quality reports

In [ ]:
def profile_dataframe(df: DataFrame, dataset_name: str, full_profile: bool = True) -> DataFrame:
    row_count = df.count()
    selected = df.columns if full_profile else [
        c for c in [
            "TransactionID", "isFraud", "TransactionDT", "TransactionAmt", "ProductCD", "card1", "card4", "card6",
            "P_emaildomain", "R_emaildomain", "DeviceType", "DeviceInfo", "dist1", "dist2", "C1", "C2", "D1", "D2",
        ] if c in df.columns
    ]
    rows: list[dict[str, object]] = []
    dtype_map = dict(df.dtypes)
    for batch in chunked(selected, PROFILE_BATCH_SIZE):
        expressions = []
        for column in batch:
            expressions.extend([
                F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(f"{column}__null"),
                F.approx_count_distinct(F.col(column)).alias(f"{column}__distinct"),
            ])
        metrics = df.agg(*expressions).first().asDict()
        for column in batch:
            null_count = int(metrics[f"{column}__null"] or 0)
            distinct = int(metrics[f"{column}__distinct"] or 0)
            rows.append({
                "dataset": dataset_name,
                "column_name": column,
                "spark_type": dtype_map.get(column, "unknown"),
                "row_count": row_count,
                "null_count": null_count,
                "null_pct": float(null_count / row_count * 100.0) if row_count else 0.0,
                "approx_distinct_count": distinct,
                "cardinality_ratio": float(distinct / row_count) if row_count else 0.0,
                "is_sparse_over_80pct": bool(row_count and null_count / row_count > 0.8),
                "is_high_cardinality_over_10pct": bool(row_count and distinct / row_count > 0.1),
            })
    return spark.createDataFrame(pd.DataFrame(rows))


train_profile = profile_dataframe(train_merged, "train", RUN_FULL_PROFILE)
test_profile = profile_dataframe(test_merged, "test", RUN_FULL_PROFILE)
data_profile = train_profile.unionByName(test_profile)
data_profile.orderBy(F.desc("null_pct")).show(30, truncate=False)

class_distribution = (
    train_merged.groupBy("isFraud")
    .agg(F.count("*").alias("transaction_count"))
    .withColumn("percentage", F.round(F.col("transaction_count") / F.sum("transaction_count").over(Window.partitionBy()) * 100, 6))
    .orderBy("isFraud")
)
class_distribution.show()

fraud_count = train_merged.filter(F.col("isFraud") == 1).count()
legit_count = train_merged.filter(F.col("isFraud") == 0).count()
imbalance_summary = {
    "fraud_count": fraud_count,
    "legitimate_count": legit_count,
    "fraud_rate": fraud_count / train_rows,
    "legitimate_to_fraud_ratio": legit_count / max(fraud_count, 1),
}
print(json.dumps(imbalance_summary, indent=2))

write_parquet(data_profile, REPORTS_DIR / "data_profile_parquet")
write_single_csv(data_profile.orderBy(F.desc("null_pct")), REPORTS_DIR / "data_profile_csv")
write_single_csv(key_audit, REPORTS_DIR / "key_audit_csv")
write_single_csv(join_audit, REPORTS_DIR / "join_audit_csv")
write_single_csv(class_distribution, REPORTS_DIR / "class_distribution_csv")
write_json(imbalance_summary, REPORTS_DIR / "imbalance_summary.json")

## Cleaning and deterministic feature engineering

In [ ]:
CATEGORICAL_TO_NORMALIZE = [
    "ProductCD", "card1", "card2", "card3", "card4", "card5", "card6", "addr1", "addr2",
    "P_emaildomain", "R_emaildomain", "DeviceType", "DeviceInfo", "M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9",
    "id_12", "id_15", "id_16", "id_23", "id_27", "id_28", "id_29", "id_30", "id_31", "id_33", "id_34", "id_35", "id_36", "id_37", "id_38",
]


def normalize_categoricals(df: DataFrame) -> DataFrame:
    result = df
    for column in CATEGORICAL_TO_NORMALIZE:
        if column in result.columns:
            result = result.withColumn(
                column,
                F.when(F.trim(F.col(column).cast("string")) == "", F.lit(None))
                 .otherwise(F.lower(F.trim(F.col(column).cast("string")))),
            )
    return result


def add_base_features(df: DataFrame) -> DataFrame:
    identity_columns = [c for c in df.columns if c.startswith("id_")]
    selected_missing_columns = [
        c for c in [
            "TransactionAmt", "ProductCD", "card1", "card2", "card3", "card4", "card5", "card6",
            "addr1", "addr2", "P_emaildomain", "R_emaildomain", "DeviceType", "DeviceInfo",
            "dist1", "dist2", "C1", "C2", "C3", "D1", "D2", "D3",
        ] if c in df.columns
    ]
    selected_missing_expr = reduce(
        lambda left, right: left + right,
        [F.when(F.col(c).isNull(), F.lit(1)).otherwise(F.lit(0)) for c in selected_missing_columns],
        F.lit(0),
    )
    identity_missing_expr = reduce(
        lambda left, right: left + right,
        [F.when(F.col(c).isNull(), F.lit(1)).otherwise(F.lit(0)) for c in identity_columns],
        F.lit(0),
    ) if identity_columns else F.lit(0)

    result = (
        df
        .withColumn("transaction_day", F.floor(F.col("TransactionDT") / F.lit(86400)).cast("long"))
        .withColumn("transaction_week", F.floor(F.col("TransactionDT") / F.lit(604800)).cast("long"))
        .withColumn("transaction_hour", F.floor((F.col("TransactionDT") % F.lit(86400)) / F.lit(3600)).cast("int"))
        .withColumn("transaction_period", F.concat(F.lit("week_"), F.col("transaction_week").cast("string")))
        .withColumn("log_transaction_amount", F.log1p(F.col("TransactionAmt").cast("double")))
        .withColumn("amount_decimal", (F.col("TransactionAmt") - F.floor(F.col("TransactionAmt"))).cast("double"))
        .withColumn("amount_band", F.when(F.col("TransactionAmt") <= 10, "00_0_10")
                    .when(F.col("TransactionAmt") <= 25, "01_10_25")
                    .when(F.col("TransactionAmt") <= 50, "02_25_50")
                    .when(F.col("TransactionAmt") <= 100, "03_50_100")
                    .when(F.col("TransactionAmt") <= 250, "04_100_250")
                    .when(F.col("TransactionAmt") <= 500, "05_250_500")
                    .when(F.col("TransactionAmt") <= 1000, "06_500_1000")
                    .otherwise("07_1000_plus"))
        .withColumn("selected_missing_count", selected_missing_expr.cast("int"))
        .withColumn("selected_missing_ratio", (F.col("selected_missing_count") / F.lit(max(len(selected_missing_columns), 1))).cast("double"))
        .withColumn("identity_missing_count", identity_missing_expr.cast("int"))
        .withColumn("has_device_info", F.when(F.col("DeviceInfo").isNotNull(), 1).otherwise(0).cast("int"))
        .withColumn("has_p_email", F.when(F.col("P_emaildomain").isNotNull(), 1).otherwise(0).cast("int"))
        .withColumn("has_r_email", F.when(F.col("R_emaildomain").isNotNull(), 1).otherwise(0).cast("int"))
        .withColumn("same_email_domain", F.when(F.coalesce(F.col("P_emaildomain"), F.lit("__NA__")) == F.coalesce(F.col("R_emaildomain"), F.lit("__NA__")), 1).otherwise(0).cast("int"))
        .withColumn("device_family", F.when(F.lower(F.col("DeviceInfo")).rlike("iphone|ipad|ios"), "apple")
                    .when(F.lower(F.col("DeviceInfo")).rlike("android|samsung|sm-"), "android")
                    .when(F.lower(F.col("DeviceInfo")).rlike("windows"), "windows")
                    .when(F.lower(F.col("DeviceInfo")).rlike("mac"), "mac")
                    .otherwise("other"))
        .withColumn("card_entity_key", F.concat_ws("|", *[F.coalesce(F.col(c).cast("string"), F.lit("__NA__")) for c in ["card1", "card2", "card3", "card5"] if c in df.columns]))
        .withColumn("email_entity_key", F.coalesce(F.col("P_emaildomain"), F.lit("__NA__")))
        .withColumn("device_entity_key", F.concat_ws("|", F.coalesce(F.col("DeviceType"), F.lit("__NA__")), F.coalesce(F.col("DeviceInfo"), F.lit("__NA__"))))
    )
    return result


clean_train = add_base_features(normalize_categoricals(train_merged))
clean_test = add_base_features(normalize_categoricals(test_merged))
high_amount_threshold = clean_train.approxQuantile("TransactionAmt", [0.99], 0.001)[0]
clean_train = clean_train.withColumn("high_amount_flag", F.when(F.col("TransactionAmt") >= high_amount_threshold, 1).otherwise(0).cast("int"))
clean_test = clean_test.withColumn("high_amount_flag", F.when(F.col("TransactionAmt") >= high_amount_threshold, 1).otherwise(0).cast("int"))
print("99th percentile transaction amount:", high_amount_threshold)

## Risk-focused EDA with Spark SQL

In [ ]:
clean_train.createOrReplaceTempView("train_clean")

eda_queries = {
    "fraud_overview": """
        SELECT COUNT(*) AS transactions,
               SUM(CASE WHEN isFraud = 1 THEN 1 ELSE 0 END) AS fraud_transactions,
               ROUND(AVG(isFraud) * 100, 6) AS fraud_rate_pct,
               ROUND(AVG(TransactionAmt), 4) AS avg_amount
        FROM train_clean
    """,
    "fraud_by_product": """
        SELECT COALESCE(ProductCD, '__missing__') AS ProductCD,
               COUNT(*) AS transactions,
               ROUND(AVG(isFraud) * 100, 6) AS fraud_rate_pct,
               ROUND(AVG(TransactionAmt), 4) AS avg_amount
        FROM train_clean
        GROUP BY COALESCE(ProductCD, '__missing__')
        ORDER BY fraud_rate_pct DESC
    """,
    "fraud_by_amount_band": """
        SELECT amount_band, COUNT(*) AS transactions,
               ROUND(AVG(isFraud) * 100, 6) AS fraud_rate_pct
        FROM train_clean
        GROUP BY amount_band
        ORDER BY amount_band
    """,
    "fraud_by_device": """
        SELECT COALESCE(DeviceType, '__missing__') AS DeviceType,
               COALESCE(device_family, '__missing__') AS device_family,
               COUNT(*) AS transactions,
               ROUND(AVG(isFraud) * 100, 6) AS fraud_rate_pct
        FROM train_clean
        GROUP BY COALESCE(DeviceType, '__missing__'), COALESCE(device_family, '__missing__')
        HAVING COUNT(*) >= 100
        ORDER BY fraud_rate_pct DESC
    """,
    "fraud_by_email": """
        SELECT COALESCE(P_emaildomain, '__missing__') AS P_emaildomain,
               COUNT(*) AS transactions,
               ROUND(AVG(isFraud) * 100, 6) AS fraud_rate_pct
        FROM train_clean
        GROUP BY COALESCE(P_emaildomain, '__missing__')
        HAVING COUNT(*) >= 100
        ORDER BY fraud_rate_pct DESC, transactions DESC
        LIMIT 30
    """,
    "fraud_by_hour": """
        SELECT transaction_hour, COUNT(*) AS transactions,
               ROUND(AVG(isFraud) * 100, 6) AS fraud_rate_pct,
               ROUND(AVG(TransactionAmt), 4) AS avg_amount
        FROM train_clean
        GROUP BY transaction_hour
        ORDER BY transaction_hour
    """,
    "fraud_by_week": """
        SELECT transaction_week, COUNT(*) AS transactions,
               ROUND(AVG(isFraud) * 100, 6) AS fraud_rate_pct
        FROM train_clean
        GROUP BY transaction_week
        ORDER BY transaction_week
    """,
}

eda_results: dict[str, DataFrame] = {}
for name, query in eda_queries.items():
    result = spark.sql(query)
    eda_results[name] = result
    print(f"\n--- {name} ---")
    result.show(30, truncate=False)
    write_single_csv(result, REPORTS_DIR / f"{name}_csv")

# Only aggregated Spark results are converted to Pandas for figures.
class_pd = class_distribution.toPandas()
ax = class_pd.plot.bar(x="isFraud", y="transaction_count", legend=False, title="IEEE-CIS Class Distribution")
ax.set_xlabel("isFraud")
ax.set_ylabel("Transactions")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "class_distribution.png", dpi=160)
plt.close()

product_pd = eda_results["fraud_by_product"].toPandas()
ax = product_pd.plot.bar(x="ProductCD", y="fraud_rate_pct", legend=False, title="Fraud Rate by ProductCD")
ax.set_ylabel("Fraud rate (%)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fraud_rate_by_product.png", dpi=160)
plt.close()

week_pd = eda_results["fraud_by_week"].toPandas()
ax = week_pd.plot.line(x="transaction_week", y="fraud_rate_pct", legend=False, title="Fraud Rate over Transaction Weeks")
ax.set_ylabel("Fraud rate (%)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fraud_rate_by_week.png", dpi=160)
plt.close()

## Chronological split and leakage-safe aggregate features

In [ ]:
q70, q85 = clean_train.approxQuantile("TransactionDT", [0.70, 0.85], 0.001)
train_split_base = clean_train.filter(F.col("TransactionDT") <= F.lit(q70))
validation_split_base = clean_train.filter((F.col("TransactionDT") > F.lit(q70)) & (F.col("TransactionDT") <= F.lit(q85)))
holdout_split_base = clean_train.filter(F.col("TransactionDT") > F.lit(q85))

split_counts = {
    "train": train_split_base.count(),
    "validation": validation_split_base.count(),
    "holdout": holdout_split_base.count(),
    "q70_transaction_dt": q70,
    "q85_transaction_dt": q85,
}
print(json.dumps(split_counts, indent=2))
assert sum(split_counts[k] for k in ["train", "validation", "holdout"]) == train_rows


def build_training_lookups(train_df: DataFrame) -> dict[str, DataFrame]:
    return {
        "card": train_df.groupBy("card_entity_key").agg(
            F.count("*").alias("card_history_count"),
            F.sum("TransactionAmt").alias("card_history_amount_sum"),
            F.max("TransactionDT").alias("card_last_transaction_dt"),
        ),
        "email": train_df.groupBy("email_entity_key").agg(
            F.count("*").alias("email_history_count"),
            F.sum("TransactionAmt").alias("email_history_amount_sum"),
        ),
        "device": train_df.groupBy("device_entity_key").agg(
            F.count("*").alias("device_history_count"),
            F.sum("TransactionAmt").alias("device_history_amount_sum"),
        ),
    }


def add_training_window_history(df: DataFrame) -> DataFrame:
    card_order = Window.partitionBy("card_entity_key").orderBy("TransactionDT", "TransactionID")
    card_history = card_order.rowsBetween(Window.unboundedPreceding, -1)
    email_history = Window.partitionBy("email_entity_key").orderBy("TransactionDT", "TransactionID").rowsBetween(Window.unboundedPreceding, -1)
    device_history = Window.partitionBy("device_entity_key").orderBy("TransactionDT", "TransactionID").rowsBetween(Window.unboundedPreceding, -1)
    return (
        df
        .withColumn("prior_card_transaction_count", F.count(F.lit(1)).over(card_history).cast("long"))
        .withColumn("prior_card_amount_sum", F.coalesce(F.sum("TransactionAmt").over(card_history), F.lit(0.0)).cast("double"))
        .withColumn("prior_card_avg_amount", F.when(F.col("prior_card_transaction_count") > 0, F.col("prior_card_amount_sum") / F.col("prior_card_transaction_count")).otherwise(F.lit(0.0)))
        .withColumn("time_since_previous_card_transaction", (F.col("TransactionDT") - F.lag("TransactionDT").over(card_order)).cast("double"))
        .withColumn("prior_email_transaction_count", F.count(F.lit(1)).over(email_history).cast("long"))
        .withColumn("prior_email_amount_sum", F.coalesce(F.sum("TransactionAmt").over(email_history), F.lit(0.0)).cast("double"))
        .withColumn("prior_device_transaction_count", F.count(F.lit(1)).over(device_history).cast("long"))
        .withColumn("prior_device_amount_sum", F.coalesce(F.sum("TransactionAmt").over(device_history), F.lit(0.0)).cast("double"))
    )


def apply_training_lookups(df: DataFrame, lookups: dict[str, DataFrame]) -> DataFrame:
    return (
        df
        .join(F.broadcast(lookups["email"]), on="email_entity_key", how="left")
        .join(lookups["card"], on="card_entity_key", how="left")
        .join(lookups["device"], on="device_entity_key", how="left")
        .withColumn("prior_card_transaction_count", F.coalesce(F.col("card_history_count"), F.lit(0)).cast("long"))
        .withColumn("prior_card_amount_sum", F.coalesce(F.col("card_history_amount_sum"), F.lit(0.0)).cast("double"))
        .withColumn("prior_card_avg_amount", F.when(F.col("prior_card_transaction_count") > 0, F.col("prior_card_amount_sum") / F.col("prior_card_transaction_count")).otherwise(F.lit(0.0)))
        .withColumn("time_since_previous_card_transaction", (F.col("TransactionDT") - F.col("card_last_transaction_dt")).cast("double"))
        .withColumn("prior_email_transaction_count", F.coalesce(F.col("email_history_count"), F.lit(0)).cast("long"))
        .withColumn("prior_email_amount_sum", F.coalesce(F.col("email_history_amount_sum"), F.lit(0.0)).cast("double"))
        .withColumn("prior_device_transaction_count", F.coalesce(F.col("device_history_count"), F.lit(0)).cast("long"))
        .withColumn("prior_device_amount_sum", F.coalesce(F.col("device_history_amount_sum"), F.lit(0.0)).cast("double"))
        .drop("card_history_count", "card_history_amount_sum", "card_last_transaction_dt", "email_history_count", "email_history_amount_sum", "device_history_count", "device_history_amount_sum")
    )


lookups = build_training_lookups(train_split_base)
train_features = add_training_window_history(train_split_base)
validation_features = apply_training_lookups(validation_split_base, lookups)
holdout_features = apply_training_lookups(holdout_split_base, lookups)
test_features = apply_training_lookups(clean_test, lookups)

if WRITE_WIDE_FEATURE_STORE:
    write_parquet(train_features, FEATURE_STORE_DIR / "train_wide", ["transaction_period"])
    write_parquet(validation_features, FEATURE_STORE_DIR / "validation_wide", ["transaction_period"])
    write_parquet(holdout_features, FEATURE_STORE_DIR / "holdout_wide", ["transaction_period"])
    write_parquet(test_features, FEATURE_STORE_DIR / "kaggle_test_wide", ["transaction_period"])

## Model-ready tables, missing values, and class imbalance treatment

In [ ]:
NUMERIC_CANDIDATES = [
    "TransactionAmt", "log_transaction_amount", "amount_decimal", "TransactionDT", "transaction_day", "transaction_week", "transaction_hour",
    "selected_missing_count", "selected_missing_ratio", "identity_missing_count", "has_identity", "has_device_info", "has_p_email", "has_r_email",
    "same_email_domain", "high_amount_flag", "dist1", "dist2",
    "C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C10", "C11", "C12", "C13", "C14",
    "D1", "D2", "D3", "D4", "D5", "D10", "D15",
    "prior_card_transaction_count", "prior_card_amount_sum", "prior_card_avg_amount", "time_since_previous_card_transaction",
    "prior_email_transaction_count", "prior_email_amount_sum",
    "prior_device_transaction_count", "prior_device_amount_sum",
]
CATEGORICAL_CANDIDATES = ["ProductCD", "card4", "card6", "DeviceType", "device_family", "M4", "amount_band"]
NUMERIC_COLUMNS = [c for c in NUMERIC_CANDIDATES if c in train_features.columns]
CATEGORICAL_COLUMNS = [c for c in CATEGORICAL_CANDIDATES if c in train_features.columns]


def select_model_columns(df: DataFrame, include_label: bool) -> DataFrame:
    selected = ["TransactionID"]
    if include_label and "isFraud" in df.columns:
        selected.append("isFraud")
    selected += NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
    result = df.select(*selected)
    for column in NUMERIC_COLUMNS:
        result = result.withColumn(column, F.col(column).cast("double"))
    result = result.fillna("__MISSING__", subset=CATEGORICAL_COLUMNS)
    return result


train_model_raw = select_model_columns(train_features, include_label=True)
validation_model_raw = select_model_columns(validation_features, include_label=True)
holdout_model_raw = select_model_columns(holdout_features, include_label=True)
test_model_raw = select_model_columns(test_features, include_label=False)

imputed_names = [f"{c}__imputed" for c in NUMERIC_COLUMNS]
imputer = Imputer(strategy="median", inputCols=NUMERIC_COLUMNS, outputCols=imputed_names)
imputer_model = imputer.fit(train_model_raw)


def apply_imputer(df: DataFrame) -> DataFrame:
    transformed = imputer_model.transform(df)
    identity_columns = ["TransactionID"] + (["isFraud"] if "isFraud" in transformed.columns else [])
    return transformed.select(
        *identity_columns,
        *[F.col(f"{c}__imputed").alias(c) for c in NUMERIC_COLUMNS],
        *CATEGORICAL_COLUMNS,
    )


train_model_ready = apply_imputer(train_model_raw).persist(StorageLevel.MEMORY_AND_DISK)
validation_model_ready = apply_imputer(validation_model_raw).persist(StorageLevel.MEMORY_AND_DISK)
holdout_model_ready = apply_imputer(holdout_model_raw).persist(StorageLevel.MEMORY_AND_DISK)
test_model_ready = apply_imputer(test_model_raw).persist(StorageLevel.MEMORY_AND_DISK)

train_class_counts = {int(row["isFraud"]): int(row["count"]) for row in train_model_ready.groupBy("isFraud").count().collect()}
train_legit = train_class_counts.get(0, 0)
train_fraud = train_class_counts.get(1, 0)
train_total = train_legit + train_fraud
fraud_weight = train_total / (2.0 * max(train_fraud, 1))
legit_weight = train_total / (2.0 * max(train_legit, 1))
train_weighted = train_model_ready.withColumn(
    "class_weight",
    F.when(F.col("isFraud") == 1, F.lit(fraud_weight)).otherwise(F.lit(legit_weight)).cast("double"),
)

fraud_train = train_model_ready.filter(F.col("isFraud") == 1)
legit_train = train_model_ready.filter(F.col("isFraud") == 0)
desired_legit = min(train_legit, int(train_fraud * IMBALANCE_RATIO))
legit_fraction = min(1.0, desired_legit / max(train_legit, 1))
legit_sample = legit_train.sample(withReplacement=False, fraction=legit_fraction, seed=SEED)
train_balanced = fraud_train.unionByName(legit_sample).repartition(max(8, spark.sparkContext.defaultParallelism)).persist(StorageLevel.MEMORY_AND_DISK)

def label_counts(df: DataFrame) -> tuple[int, int]:
    counts = {int(row["isFraud"]): int(row["count"]) for row in df.groupBy("isFraud").count().collect()}
    return counts.get(0, 0), counts.get(1, 0)

balanced_legit, balanced_fraud = label_counts(train_balanced)
validation_legit, validation_fraud = label_counts(validation_model_ready)
holdout_legit, holdout_fraud = label_counts(holdout_model_ready)
balance_report = spark.createDataFrame(pd.DataFrame([
    {"dataset": "train_original", "legitimate": train_legit, "fraud": train_fraud, "legit_to_fraud_ratio": train_legit / max(train_fraud, 1)},
    {"dataset": "train_balanced", "legitimate": balanced_legit, "fraud": balanced_fraud, "legit_to_fraud_ratio": balanced_legit / max(balanced_fraud, 1)},
    {"dataset": "validation_untouched", "legitimate": validation_legit, "fraud": validation_fraud, "legit_to_fraud_ratio": validation_legit / max(validation_fraud, 1)},
    {"dataset": "holdout_untouched", "legitimate": holdout_legit, "fraud": holdout_fraud, "legit_to_fraud_ratio": holdout_legit / max(holdout_fraud, 1)},
]))
balance_report.show(truncate=False)

write_parquet(train_model_ready, MODEL_READY_DIR / "train_original")
write_parquet(train_weighted, MODEL_READY_DIR / "train_weighted")
write_parquet(train_balanced, MODEL_READY_DIR / "train_balanced")
write_parquet(validation_model_ready, MODEL_READY_DIR / "validation")
write_parquet(holdout_model_ready, MODEL_READY_DIR / "holdout")
write_parquet(test_model_ready, MODEL_READY_DIR / "kaggle_test")
write_single_csv(balance_report, REPORTS_DIR / "class_balance_report_csv")

## Spark MLlib Decision Tree demonstration and demo cases

In [ ]:
def calculate_binary_metrics(predictions: DataFrame, dataset_name: str) -> dict[str, object]:
    counts = {
        (int(row["isFraud"]), int(row["prediction"])): int(row["count"])
        for row in predictions.groupBy("isFraud", "prediction").count().collect()
    }
    tp = counts.get((1, 1), 0)
    tn = counts.get((0, 0), 0)
    fp = counts.get((0, 1), 0)
    fn = counts.get((1, 0), 0)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    pr_auc = BinaryClassificationEvaluator(labelCol="isFraud", rawPredictionCol="rawPrediction", metricName="areaUnderPR").evaluate(predictions)
    roc_auc = BinaryClassificationEvaluator(labelCol="isFraud", rawPredictionCol="rawPrediction", metricName="areaUnderROC").evaluate(predictions)
    return {
        "dataset": dataset_name,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
        "precision_fraud": precision,
        "recall_fraud": recall,
        "f1_fraud": f1,
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
    }


model_metrics: list[dict[str, object]] = []
if RUN_MODEL_DEMO:
    indexers = [
        StringIndexer(inputCol=c, outputCol=f"{c}__idx", handleInvalid="keep")
        for c in CATEGORICAL_COLUMNS
    ]
    assembled_inputs = NUMERIC_COLUMNS + [f"{c}__idx" for c in CATEGORICAL_COLUMNS]
    assembler = VectorAssembler(inputCols=assembled_inputs, outputCol="features", handleInvalid="keep")
    classifier = DecisionTreeClassifier(
        labelCol="isFraud",
        featuresCol="features",
        predictionCol="prediction",
        probabilityCol="probability",
        rawPredictionCol="rawPrediction",
        maxDepth=int(os.getenv("DT_MAX_DEPTH", "8")),
        maxBins=int(os.getenv("DT_MAX_BINS", "128")),
        minInstancesPerNode=int(os.getenv("DT_MIN_INSTANCES_PER_NODE", "50")),
        seed=SEED,
    )
    ml_pipeline = Pipeline(stages=[*indexers, assembler, classifier])
    start = time.time()
    ml_model = ml_pipeline.fit(train_balanced)
    training_seconds = time.time() - start
    if MODEL_DIR.exists():
        import shutil
        shutil.rmtree(MODEL_DIR)
    ml_model.write().overwrite().save(spark_path(MODEL_DIR))

    validation_predictions = ml_model.transform(validation_model_ready).withColumn("fraud_probability", vector_to_array("probability")[1])
    holdout_predictions = ml_model.transform(holdout_model_ready).withColumn("fraud_probability", vector_to_array("probability")[1])
    test_predictions = ml_model.transform(test_model_ready).withColumn("fraud_probability", vector_to_array("probability")[1])

    model_metrics.append({**calculate_binary_metrics(validation_predictions, "validation"), "training_seconds": training_seconds})
    model_metrics.append({**calculate_binary_metrics(holdout_predictions, "holdout"), "training_seconds": training_seconds})
    model_metrics_df = spark.createDataFrame(pd.DataFrame(model_metrics))
    model_metrics_df.show(truncate=False)
    write_single_csv(model_metrics_df, REPORTS_DIR / "decision_tree_metrics_csv")
    write_json({"metrics": model_metrics, "training_seconds": training_seconds}, REPORTS_DIR / "decision_tree_metrics.json")

    case_type = (
        F.when((F.col("isFraud") == 1) & (F.col("prediction") == 1), "true_positive")
        .when((F.col("isFraud") == 0) & (F.col("prediction") == 0), "true_negative")
        .when((F.col("isFraud") == 0) & (F.col("prediction") == 1), "false_positive")
        .otherwise("false_negative")
    )
    case_window = Window.partitionBy("case_type").orderBy(F.desc("fraud_probability"), F.asc("TransactionID"))
    demo_cases = (
        holdout_predictions
        .withColumn("case_type", case_type)
        .withColumn("case_rank", F.row_number().over(case_window))
        .filter(F.col("case_rank") <= 3)
        .select(
            "case_type", "case_rank", "TransactionID", "isFraud", "prediction", "fraud_probability",
            "TransactionAmt", "ProductCD", "card4", "card6", "DeviceType", "device_family", "transaction_hour",
            "prior_card_transaction_count", "prior_email_transaction_count", "prior_device_transaction_count",
        )
        .orderBy("case_type", "case_rank")
    )
    demo_cases.show(20, truncate=False)
    write_parquet(demo_cases, DEMO_DIR / "demo_cases_parquet")
    write_single_csv(demo_cases, DEMO_DIR / "demo_cases_csv")

    kaggle_predictions = test_predictions.select("TransactionID", F.col("fraud_probability").alias("isFraud")).orderBy("TransactionID")
    write_single_csv(kaggle_predictions, DEMO_DIR / "kaggle_test_predictions_csv")
else:
    logger.info("RUN_MODEL_DEMO=false; skipped the Decision Tree demonstration.")

## Manifest, training handover, and completion checks

In [ ]:
feature_catalog_rows = []
for column in NUMERIC_COLUMNS:
    feature_catalog_rows.append({
        "feature_name": column,
        "feature_type": "numeric",
        "missing_value_policy": "median fitted on chronological training split",
        "training_ready": True,
    })
for column in CATEGORICAL_COLUMNS:
    feature_catalog_rows.append({
        "feature_name": column,
        "feature_type": "categorical",
        "missing_value_policy": "__MISSING__ category",
        "training_ready": True,
    })
feature_catalog = spark.createDataFrame(pd.DataFrame(feature_catalog_rows))
write_single_csv(feature_catalog, REPORTS_DIR / "feature_catalog_csv")

manifest = {
    "pipeline": "BDA501 IEEE-CIS Spark preprocessing and EDA",
    "generated_at_utc": pd.Timestamp.utcnow().isoformat(),
    "platform": platform.platform(),
    "python_version": sys.version,
    "spark_version": spark.version,
    "spark_master": spark.sparkContext.master,
    "project_root": str(PROJECT_ROOT),
    "raw_data_dir": str(RAW_DATA_DIR),
    "output_dir": str(OUTPUT_DIR),
    "source_files": inventory_pdf.to_dict(orient="records"),
    "split_boundaries": {"q70_transaction_dt": q70, "q85_transaction_dt": q85},
    "split_counts": split_counts,
    "imbalance": imbalance_summary,
    "class_weights": {"legitimate": legit_weight, "fraud": fraud_weight},
    "undersampling_target_legitimate_to_fraud_ratio": IMBALANCE_RATIO,
    "numeric_features": NUMERIC_COLUMNS,
    "categorical_features": CATEGORICAL_COLUMNS,
    "outputs": {
        "train_original": str(MODEL_READY_DIR / "train_original"),
        "train_weighted": str(MODEL_READY_DIR / "train_weighted"),
        "train_balanced": str(MODEL_READY_DIR / "train_balanced"),
        "validation": str(MODEL_READY_DIR / "validation"),
        "holdout": str(MODEL_READY_DIR / "holdout"),
        "kaggle_test": str(MODEL_READY_DIR / "kaggle_test"),
        "reports": str(REPORTS_DIR),
        "demo": str(DEMO_DIR),
        "model": str(MODEL_DIR) if RUN_MODEL_DEMO else None,
    },
    "model_demo_metrics": model_metrics,
}
write_json(manifest, OUTPUT_DIR / "manifest.json")

required_output_paths = [
    MODEL_READY_DIR / "train_original",
    MODEL_READY_DIR / "train_weighted",
    MODEL_READY_DIR / "train_balanced",
    MODEL_READY_DIR / "validation",
    MODEL_READY_DIR / "holdout",
    MODEL_READY_DIR / "kaggle_test",
    OUTPUT_DIR / "manifest.json",
]
missing_outputs = [str(path) for path in required_output_paths if not path.exists()]
assert not missing_outputs, f"Missing required outputs: {missing_outputs}"

print("\nPIPELINE COMPLETED SUCCESSFULLY")
print("Training-ready data:", MODEL_READY_DIR)
print("Manifest:", OUTPUT_DIR / "manifest.json")
print("Reports:", REPORTS_DIR)
print("Demo cases:", DEMO_DIR)
print("Decision Tree model:", MODEL_DIR if RUN_MODEL_DEMO else "skipped")


## Docker handover

The supplied Docker files run the same pipeline non-interactively. From the project root:

```powershell
docker compose -f docker-compose.preprocessing.yml build
docker compose -f docker-compose.preprocessing.yml run --rm preprocess
```

Processed Parquet data remains on the Windows host under:

```text
D:\MSE\16. Big Data\Fraud-Detection-Score-Risk\data\processed\ieee_cis_spark
```

The training service can mount that directory read-only as `/app/data/processed/ieee_cis_spark`.
